# Notebook 03: Pseudo-Label Generation & Supervised Classification

This notebook handles:
1. Loading clustering results from Notebook 02
2. Aligning pseudo-labels with real labels
3. Train/test split
4. Hyperparameter tuning with GridSearchCV
5. Training final Logistic Regression model
6. Saving results for evaluation

In [11]:
import pandas as pd
import numpy as np
import sys
import os

# Get the absolute path to the directory where THIS script is located
# Then go to the 'src' folder relative to that
current_dir = os.path.dirname(os.path.abspath('')) # Use os.getcwd() if in a Notebook
src_path = os.path.join(current_dir, 'src')

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from evaluation import align_pseudo_labels
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression

# Set random seed
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


## Phase 1: Load Clustering Results

In [12]:
# Load data from previous notebooks
cleaned_df = pd.read_csv('./outputs/cleaned_reviews.csv')
real_labels = cleaned_df['label'].values

# Load clustering results
cluster_labels = np.load('./outputs/best_cluster_labels.npy')
X_pca = np.load('./outputs/best_X_pca.npy')

print(f"Loaded real labels: {len(real_labels)}")
print(f"Loaded cluster labels: {len(cluster_labels)}")
print(f"Loaded PCA features: {X_pca.shape}")
print(f"Real label distribution: {np.bincount(real_labels)}")
print(f"Cluster label distribution: {np.bincount(cluster_labels)}")

Loaded real labels: 50000
Loaded cluster labels: 50000
Loaded PCA features: (50000, 20)
Real label distribution: [25000 25000]
Cluster label distribution: [30277 19723]


## Phase 2: Align Pseudo-Labels with Real Labels

In [13]:
# Align pseudo-labels (cluster labels may be flipped)
aligned_pseudo_labels, is_flipped, alignment_accuracy = align_pseudo_labels(cluster_labels, real_labels)

print("\nPSEUDO-LABEL ALIGNMENT:")
print("="*80)
print(f"Original cluster labels accuracy: {(np.sum(real_labels == cluster_labels) / len(real_labels)):.4f}")
print(f"Flipped cluster labels accuracy: {(np.sum(real_labels == (1-cluster_labels)) / len(real_labels)):.4f}")
print(f"\nBest alignment: {'FLIPPED' if is_flipped else 'NORMAL'}")
print(f"Alignment accuracy: {alignment_accuracy:.4f} ({alignment_accuracy*100:.2f}%)")
print(f"\nPseudo-label distribution: {np.bincount(aligned_pseudo_labels)}")
print("="*80)


PSEUDO-LABEL ALIGNMENT:
Original cluster labels accuracy: 0.2919
Flipped cluster labels accuracy: 0.7081

Best alignment: FLIPPED
Alignment accuracy: 0.7081 (70.81%)

Pseudo-label distribution: [19723 30277]


## Phase 3: Train/Test Split

In [14]:
# Split data for supervised learning (using pseudo-labels for training)
X_train, X_test, y_train_pseudo, y_test_real = train_test_split(
    X_pca,
    aligned_pseudo_labels,
    test_size=0.2,
    random_state=42,
    stratify=aligned_pseudo_labels
)

# Also get real labels for test set
_, X_test_idx, _, y_test_real = train_test_split(
    range(len(real_labels)),
    real_labels,
    test_size=0.2,
    random_state=42,
    stratify=real_labels
)
y_test_real = real_labels[X_test_idx]

print("\nTRAIN/TEST SPLIT:")
print("="*80)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining label distribution (pseudo-labels): {np.bincount(y_train_pseudo)}")
print(f"Test label distribution (real labels): {np.bincount(y_test_real)}")
print("="*80)


TRAIN/TEST SPLIT:
Training set size: 40000 samples
Test set size: 10000 samples

Training label distribution (pseudo-labels): [15778 24222]
Test label distribution (real labels): [5000 5000]


## Phase 4: Hyperparameter Tuning with GridSearchCV

In [15]:
# Define hyperparameter grid
param_grid = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs'],
    'max_iter': [1000]
}

# Create base model
lr = LogisticRegression(random_state=42)

# Grid search with 5-fold cross-validation
print("Running GridSearchCV with 5-fold cross-validation...")
print("This may take a few minutes.\n")

grid_search = GridSearchCV(
    lr,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train_pseudo)

print("\n✓ GridSearchCV completed!")

Running GridSearchCV with 5-fold cross-validation...
This may take a few minutes.

Fitting 5 folds for each of 3 candidates, totalling 15 fits

✓ GridSearchCV completed!


In [16]:
# Display cross-validation results
print("\nHYPERPARAMETER VALIDATION RESULTS (K-Fold CV):")
print("="*80)

cv_results = pd.DataFrame(grid_search.cv_results_)

# Create clean table
results_table = pd.DataFrame({
    'C': cv_results['param_C'],
    'Mean CV Score': cv_results['mean_test_score'].round(4),
    'Std CV Score': cv_results['std_test_score'].round(4)
})

print(results_table.to_string(index=False))
print("="*80)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Display fold scores for best parameters
best_index = grid_search.best_index_
results = pd.DataFrame(grid_search.cv_results_)
fold_columns = [
    "split0_test_score",
    "split1_test_score",
    "split2_test_score",
    "split3_test_score",
    "split4_test_score"
]

display_columns = [
    "param_C",
    *fold_columns,
    "mean_test_score"
]

print(results[display_columns])


HYPERPARAMETER VALIDATION RESULTS (K-Fold CV):
   C  Mean CV Score  Std CV Score
 0.1         0.9361        0.0024
 1.0         0.9855        0.0017
10.0         0.9966        0.0006

Best parameters: {'C': 10, 'max_iter': 1000, 'solver': 'lbfgs'}
Best CV score: 0.9966
   param_C  split0_test_score  split1_test_score  split2_test_score  \
0      0.1           0.939856           0.935173           0.935303   
1      1.0           0.986960           0.987966           0.985072   
2     10.0           0.996498           0.997249           0.996247   

   split3_test_score  split4_test_score  mean_test_score  
0           0.932822           0.937255         0.936082  
1           0.984443           0.983309         0.985550  
2           0.997248           0.995746         0.996598  


## Phase 5: Train Final Model

In [17]:
# Get best model
best_model = grid_search.best_estimator_

print(f"\nTraining final model with best parameters...")
best_model.fit(X_train, y_train_pseudo)
print("✓ Model training completed!")

# Make predictions
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

print(f"\nTraining accuracy: {(y_pred_train == y_train_pseudo).mean():.4f}")
print(f"Test accuracy (against pseudo-labels): {(y_pred_test == y_train_pseudo[len(y_train_pseudo)-len(y_test_real):]).mean():.4f}")


Training final model with best parameters...
✓ Model training completed!

Training accuracy: 0.9975
Test accuracy (against pseudo-labels): 0.5248


## Phase 6: Save Results

In [18]:
# Save model predictions and parameters
np.save('./outputs/y_pred_test.npy', y_pred_test)
np.save('./outputs/y_test_real.npy', y_test_real)
np.save('./outputs/y_train_pseudo.npy', y_train_pseudo)
np.save('./outputs/X_train.npy', X_train)
np.save('./outputs/X_test.npy', X_test)

print("✓ Saved y_pred_test.npy")
print("✓ Saved y_test_real.npy")
print("✓ Saved y_train_pseudo.npy")
print("✓ Saved X_train.npy")
print("✓ Saved X_test.npy")

# Save hyperparameter results
results_table.to_csv('./outputs/hyperparameter_validation.csv', index=False)
print("✓ Saved hyperparameter_validation.csv")

✓ Saved y_pred_test.npy
✓ Saved y_test_real.npy
✓ Saved y_train_pseudo.npy
✓ Saved X_train.npy
✓ Saved X_test.npy
✓ Saved hyperparameter_validation.csv


In [20]:
# Save model info
model_info = {
    'best_C': grid_search.best_params_['C'],
    'best_cv_score': grid_search.best_score_,
    'n_features': X_train.shape[1],
    'n_train': len(X_train),
    'n_test': len(X_test),
    'alignment_accuracy': float(alignment_accuracy),
    'is_pseudo_labels_flipped': bool(is_flipped)
}

import json
with open('./outputs/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("✓ Saved model_info.json")

✓ Saved model_info.json


In [19]:
print("\n" + "="*80)
print("SUPERVISED LEARNING PHASE COMPLETE!")
print("="*80)
print(f"Best C parameter: {grid_search.best_params_['C']}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")
print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Pseudo-label alignment: {alignment_accuracy:.4f}")
print("\nNext step: Run Notebook 04 (Evaluation & Error Analysis)")


SUPERVISED LEARNING PHASE COMPLETE!
Best C parameter: 10
Best CV F1-Score: 0.9966

Training set size: 40000
Test set size: 10000
Pseudo-label alignment: 0.7081

Next step: Run Notebook 04 (Evaluation & Error Analysis)
